# 02 — Ajustes nos dados

**CRISP-DM fase 3 — preparação dos dados** · Meridiano Imobiliária

---

A [análise exploratória](01-analise-exploratoria.ipynb) descreveu a base **como ela
chega**: 2.930 registros, 82 colunas, 27 delas com célula vazia, escalas de qualidade
guardadas como texto, dois anos impossíveis e cinco imóveis que o próprio autor da base
recomenda descartar. Este notebook é o passo seguinte: **aplicar as correções** e gravar
a base curada que todas as fases posteriores consomem.

A regra deste notebook é uma só:

> **Nenhum dado é alterado em silêncio.** Cada ajuste declara o *motivo*, mostra a
> *evidência numérica* que o justifica e registra *quantos registros* foram tocados.


In [1]:
# Importações
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/base_rh_columns_selected.csv", sep=';')

In [3]:
nulos = df.isna().mean().mul(100).round(2)
if nulos.sum() > 0:
    print("Tratando valores nulos residuais")
    for col in df.columns[df.isna().any()]:
        valor = df[col].median() if pd.api.types.is_numeric_dtype(df[col]) else df[col].mode()[0]
        df[col] = df[col].fillna(valor)
else:
    print("Nenhum valor nulo encontrado na base de RH. A base está ok!")

Nenhum valor nulo encontrado na base de RH. A base está ok!


In [4]:
if 'idade' in df.columns:
    bins_idade = [18, 30, 40, 50, 60, 100]
    labels_idade = ['18-30', '31-40', '41-50', '51-60', '60+']
    df['faixa_etaria'] = pd.cut(df['idade'], bins=bins_idade, labels=labels_idade, right=False)

col_tempo = 'anos_na_empresa' if 'anos_na_empresa' in df.columns else [c for c in df.columns if 'empresa' in c.lower()][0]
bins_tempo = [-1, 2, 5, 10, 40]
labels_tempo = ['Até 2 anos', '3 a 5 anos', '6 a 10 anos', 'Mais de 10 anos']
df['categoria_tempo_empresa'] = pd.cut(df[col_tempo], bins=bins_tempo, labels=labels_tempo)

col_mob = [c for c in df.columns if 'anteriores' in c.lower() or 'empresas' in c.lower()]
if len(col_mob) > 0:
    df['alta_mobilidade_carreira'] = (df[col_mob[0]] > 3).astype(int)

df.to_csv("../data/processed/base_rh_tratado.csv", sep=';', index=False)
print("Base de RH tratada e salva com sucesso em '../data/processed/base_rh_tratado.csv'!")

Base de RH tratada e salva com sucesso em '../data/processed/base_rh_tratado.csv'!
